<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> 책의 보충 코드, 저자: <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>코드 저장소: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 바이트 페어 인코딩(Byte Pair Encoding, BPE) 토크나이저 처음부터 구현하기

- 이는 교육 목적으로 GPT-2에서 GPT-4, Llama 3 등과 같은 모델에서 사용되는 인기 있는 바이트 페어 인코딩(BPE) 토큰화 알고리즘을 처음부터 구현하는 독립형 노트북입니다
- 토큰화의 목적에 대한 자세한 내용은 [Chapter 2](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch02/01_main-chapter-code/ch02.ipynb)를 참조하세요. 여기의 코드는 BPE 알고리즘을 설명하는 보너스 자료입니다
- OpenAI가 원래 GPT 모델 학습을 위해 구현한 원본 BPE 토크나이저는 [여기](https://github.com/openai/gpt-2/blob/master/src/encoder.py)에서 찾을 수 있습니다
- BPE 알고리즘은 1994년에 처음 설명되었습니다: Philip Gage의 "[A New Algorithm for Data Compression](http://www.pennelynn.com/Documents/CUJ/HTML/94HTML/19940045.HTM)"
- Llama 3을 포함한 대부분의 프로젝트는 현재 계산 성능 때문에 OpenAI의 오픈 소스 [tiktoken 라이브러리](https://github.com/openai/tiktoken)를 사용합니다. 이는 예를 들어 사전 훈련된 GPT-2 및 GPT-4 토크나이저를 로드할 수 있습니다(Llama 3 모델들도 GPT-4 토크나이저를 사용하여 훈련되었습니다)
- 위의 구현들과 이 노트북에서의 제 구현의 차이점은 이것이 토크나이저를 훈련하기 위한 함수도 포함한다는 것입니다(교육 목적으로)
- 훈련을 지원하는 [minBPE](https://github.com/karpathy/minbpe)라고 하는 구현도 있으며, 이는 더 성능이 좋을 수 있습니다(제 구현은 교육 목적에 중점을 둡니다). `minbpe`와 달리 제 구현은 추가로 원본 OpenAI 토크나이저 어휘와 BPE "병합"을 로드할 수 있습니다(또한 Hugging Face 토크나이저도 다양한 토크나이저를 훈련하고 로드할 수 있습니다. 네팔어로 BPE 토크나이저를 훈련한 독자의 [이 GitHub 토론](https://github.com/rasbt/LLMs-from-scratch/discussions/485)을 참조하세요)

&nbsp;
# 1. 바이트 페어 인코딩(BPE)의 주요 아이디어

- BPE의 주요 아이디어는 LLM 훈련을 위해 텍스트를 정수 표현(토큰 ID)으로 변환하는 것입니다([Chapter 2](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch02/01_main-chapter-code/ch02.ipynb) 참조)

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/bpe-from-scratch/bpe-overview.webp" width="600px">

&nbsp;
## 1.1 비트와 바이트

- BPE 알고리즘을 다루기 전에 바이트의 개념을 소개해보겠습니다
- 텍스트를 바이트 배열로 변환하는 것을 고려해보세요(BPE는 결국 "바이트" 페어 인코딩을 의미합니다):

In [ ]:
text = "This is some text"
byte_ary = bytearray(text, "utf-8")
print(byte_ary)

- `bytearray` 객체에 대해 `list()`를 호출하면, 각 바이트는 개별 요소로 처리되고, 결과는 바이트 값에 해당하는 정수들의 리스트가 됩니다:

In [ ]:
ids = list(byte_ary)
print(ids)

- 이는 LLM의 임베딩 레이어에 필요한 토큰 ID 표현으로 텍스트를 변환하는 유효한 방법입니다
- 그러나 이 접근법의 단점은 각 문자마다 하나의 ID를 생성한다는 것입니다(짧은 텍스트에 대해서도 많은 ID가 필요합니다!)
- 즉, 17자 입력 텍스트에 대해 LLM에 입력으로 17개의 토큰 ID를 사용해야 합니다:

In [ ]:
print("문자 수(Number of characters):", len(text))
print("토큰 ID 수(Number of token IDs):", len(ids))

- 이전에 LLM을 다뤄본 적이 있다면, BPE 토크나이저가 각 문자 대신 전체 단어나 부분 단어에 대한 토큰 ID를 가진 어휘를 갖고 있다는 것을 알고 있을 것입니다
- 예를 들어, GPT-2 토크나이저는 동일한 텍스트("This is some text")를 17개가 아닌 단 4개의 토큰으로 토큰화합니다: `1212, 318, 617, 2420`
- 이는 상호작용형 [tiktoken 앱](https://tiktokenizer.vercel.app/?model=gpt2)이나 [tiktoken 라이브러리](https://github.com/openai/tiktoken)를 사용하여 다시 확인할 수 있습니다:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/bpe-from-scratch/tiktokenizer.webp" width="600px">

```python
import tiktoken

gpt2_tokenizer = tiktoken.get_encoding("gpt2")
gpt2_tokenizer.encode("This is some text")
# prints [1212, 318, 617, 2420]
```

- 바이트는 8비트로 구성되므로, 단일 바이트가 나타낼 수 있는 가능한 값은 2<sup>8</sup> = 256개이며, 0부터 255까지입니다
- `bytearray(range(0, 257))` 코드를 실행하면 `ValueError: byte must be in range(0, 256)`이라는 경고를 받게 됩니다
- BPE 토크나이저는 보통 이 256개 값을 처음 256개의 단일 문자 토큰으로 사용합니다. 다음 코드를 실행하여 시각적으로 확인할 수 있습니다:

```python
import tiktoken
gpt2_tokenizer = tiktoken.get_encoding("gpt2")

for i in range(300):
    decoded = gpt2_tokenizer.decode([i])
    print(f"{i}: {decoded}")
"""
prints:
0: !
1: "
2: #
...
255: �  # <---- 여기까지가 단일 문자 토큰
256:  t
257:  a
...
298: ent
299:  n
"""
```

- 위에서 주목할 점은 256번과 257번 항목이 단일 문자 값이 아니라 이중 문자 값(공백 + 문자)이라는 것입니다. 이는 원본 GPT-2 BPE 토크나이저의 작은 단점입니다(이는 GPT-4 토크나이저에서 개선되었습니다)

&nbsp;
## 1.2 어휘 구축하기

- BPE 토큰화 알고리즘의 목표는 `298: ent`(*entangle, entertain, enter, entrance, entity, ...* 등에서 찾을 수 있음)와 같이 흔히 발생하는 부분 단어나 다음과 같은 완전한 단어들의 어휘를 구축하는 것입니다:

```
318: is
617: some
1212: This
2420: text
```

- BPE 알고리즘은 1994년에 처음 설명되었습니다: Philip Gage의 "[A New Algorithm for Data Compression](http://www.pennelynn.com/Documents/CUJ/HTML/94HTML/19940045.HTM)"
- 실제 코드 구현에 앞서, 현재 LLM 토크나이저에 사용되는 형태는 다음 섹션들에서 설명하는 바와 같이 요약할 수 있습니다.

&nbsp;
## 1.3 BPE 알고리즘 개요

**1. 빈번한 페어 식별**
- 각 반복에서, 텍스트를 스캔하여 가장 자주 발생하는 바이트(또는 문자) 페어를 찾습니다

**2. 교체 및 기록**

- 해당 페어를 새로운 플레이스홀더 ID(아직 사용되지 않은 ID, 예: 0...255로 시작한다면 첫 번째 플레이스홀더는 256)로 교체합니다
- 이 매핑을 룩업 테이블에 기록합니다
- 룩업 테이블의 크기는 하이퍼파라미터이며, "어휘 크기"라고도 불립니다(GPT-2의 경우 50,257개입니다)

**3. 이득이 없을 때까지 반복**

- 1단계와 2단계를 계속 반복하여, 가장 빈번한 페어를 지속적으로 병합합니다
- 더 이상 압축이 불가능할 때 중단합니다(예: 한 번 이상 발생하는 페어가 없을 때)

**압축 해제(디코딩)**

- 원래 텍스트를 복원하려면, 룩업 테이블을 사용하여 각 ID를 해당하는 페어로 대체하는 과정을 역순으로 수행합니다

&nbsp;
## 1.4 BPE 알고리즘 예제

### 1.4.1 인코딩 부분의 구체적인 예제 (1.3절의 1단계 & 2단계)

- BPE 토크나이저를 위한 어휘를 구축하고 싶은 텍스트(훈련 데이터셋) `the cat in the hat`가 있다고 가정해봅시다

**반복 1**

1. 빈번한 페어 식별
  - 이 텍스트에서 "th"가 두 번 나타납니다(처음과 두 번째 "e" 앞에서)

2. 교체 및 기록
  - "th"를 아직 사용되지 않은 새로운 토큰 ID(예: 256)로 교체합니다
  - 새로운 텍스트는: `<256>e cat in <256>e hat`
  - 새로운 어휘는:

```
  0: ...
  ...
  256: "th"
```

**반복 2**

1. **빈번한 페어 식별**  
   - 텍스트 `<256>e cat in <256>e hat`에서 페어 `<256>e`가 두 번 나타납니다

2. **교체 및 기록**  
   - `<256>e`를 아직 사용되지 않은 새로운 토큰 ID(예: `257`)로 교체합니다.
   - 새로운 텍스트는:
     ```
     <257> cat in <257> hat
     ```
   - 업데이트된 어휘는:
     ```
     0: ...
     ...
     256: "th"
     257: "<256>e"
     ```

**반복 3**

1. **빈번한 페어 식별**  
   - 텍스트 `<257> cat in <257> hat`에서 페어 `<257> `가 두 번 나타납니다(처음과 "hat" 앞에서).

2. **교체 및 기록**  
   - `<257> `를 아직 사용되지 않은 새로운 토큰 ID(예: `258`)로 교체합니다.
   - 새로운 텍스트는:
     ```
     <258>cat in <258>hat
     ```
   - 업데이트된 어휘는:
     ```
     0: ...
     ...
     256: "th"
     257: "<256>e"
     258: "<257> "
     ```
     
- 그리고 계속됩니다...

&nbsp;
### 1.4.2 디코딩 부분의 구체적인 예제 (1.3절의 3단계)

- 원래 텍스트를 복원하려면, 도입된 역순으로 각 토큰 ID를 해당하는 페어로 대체하는 과정을 역순으로 수행합니다
- 최종 압축된 텍스트로 시작: `<258>cat in <258>hat`
- `<258>` → `<257> ` 대체: `<257> cat in <257> hat`
- `<257>` → `<256>e` 대체: `<256>e cat in <256>e hat`
- `<256>` → "th" 대체: `the cat in the hat`

&nbsp;
## 2. 간단한 BPE 구현

- 아래는 `tiktoken` Python 사용자 인터페이스를 모방하는 Python 클래스로서 위에서 설명한 알고리즘의 구현입니다
- 위의 인코딩 부분은 `train()`을 통한 원래 훈련 단계를 설명한다는 점에 주목하세요. 그러나 `encode()` 메소드는 비슷하게 작동합니다(특수 토큰 처리 때문에 조금 더 복잡해 보이긴 하지만):

1. 입력 텍스트를 개별 바이트로 분할
2. 인접한 토큰(페어)이 학습된 BPE 병합의 어떤 페어와 일치할 때 반복적으로 찾기 & 교체(병합)를 수행(가장 높은 것부터 낮은 "순위"로, 즉 학습된 순서대로)
3. 더 이상 병합을 적용할 수 없을 때까지 병합을 계속
4. 토큰 ID의 최종 목록이 인코딩된 출력

In [ ]:
from collections import Counter, deque
from functools import lru_cache
import json


class BPETokenizerSimple:
    def __init__(self):
        # token_id를 token_str로 매핑 (예: {11246: "some"})
        self.vocab = {}
        # token_str를 token_id로 매핑 (예: {"some": 11246})
        self.inverse_vocab = {}
        # BPE 병합 딕셔너리: {(token_id1, token_id2): merged_token_id}
        self.bpe_merges = {}

        # 공식 OpenAI GPT-2 병합의 경우, 순위 딕셔너리를 사용:
        #  {(string_A, string_B): rank} 형태, 낮은 순위 = 높은 우선순위
        self.bpe_ranks = {}

    def train(self, text, vocab_size, allowed_special={"<|endoftext|>"}):
        """
        BPE 토크나이저를 처음부터 훈련합니다.

        Args:
            text (str): 훈련 텍스트.
            vocab_size (int): 원하는 어휘 크기.
            allowed_special (set): 포함할 특수 토큰 세트.
        """

        # 전처리: 공백을 "Ġ"로 교체
        # 주목: Ġ는 GPT-2 BPE 구현의 특수성입니다
        # 예: "Hello world"가 ["Hello", "Ġworld"]로 토큰화될 수 있습니다
        # (GPT-4 BPE는 ["Hello", " world"]로 토큰화합니다)
        processed_text = []
        for i, char in enumerate(text):
            if char == " " and i != 0:
                processed_text.append("Ġ")
            if char != " ":
                processed_text.append(char)
        processed_text = "".join(processed_text)

        # "Ġ"를 포함한 고유 문자로 어휘를 초기화
        # 처음 256개의 ASCII 문자로 시작
        unique_chars = [chr(i) for i in range(256)]
        unique_chars.extend(
            char for char in sorted(set(processed_text))
            if char not in unique_chars
        )
        if "Ġ" not in unique_chars:
            unique_chars.append("Ġ")

        self.vocab = {i: char for i, char in enumerate(unique_chars)}
        self.inverse_vocab = {char: i for i, char in self.vocab.items()}

        # 허용된 특수 토큰 추가
        if allowed_special:
            for token in allowed_special:
                if token not in self.inverse_vocab:
                    new_id = len(self.vocab)
                    self.vocab[new_id] = token
                    self.inverse_vocab[token] = new_id

        # processed_text를 토큰 ID로 토큰화
        token_ids = [self.inverse_vocab[char] for char in processed_text]

        # BPE 1-3단계: 빈번한 페어를 반복적으로 찾아 교체
        for new_id in range(len(self.vocab), vocab_size):
            pair_id = self.find_freq_pair(token_ids, mode="most")
            if pair_id is None:
                break
            token_ids = self.replace_pair(token_ids, pair_id, new_id)
            self.bpe_merges[pair_id] = new_id

        # 병합된 토큰으로 어휘 구축
        for (p0, p1), new_id in self.bpe_merges.items():
            merged_token = self.vocab[p0] + self.vocab[p1]
            self.vocab[new_id] = merged_token
            self.inverse_vocab[merged_token] = new_id

    def load_vocab_and_merges_from_openai(self, vocab_path, bpe_merges_path):
        """
        OpenAI의 GPT-2 파일에서 사전 훈련된 어휘와 BPE 병합을 로드합니다.

        Args:
            vocab_path (str): 어휘 파일 경로 (GPT-2에서는 'encoder.json'이라고 함).
            bpe_merges_path (str): bpe_merges 파일 경로 (GPT-2에서는 'vocab.bpe'라고 함).
        """
        # 어휘 로드
        with open(vocab_path, "r", encoding="utf-8") as file:
            loaded_vocab = json.load(file)
            # 로드된 어휘를 올바른 형식으로 변환
            self.vocab = {int(v): k for k, v in loaded_vocab.items()}
            self.inverse_vocab = {k: int(v) for k, v in loaded_vocab.items()}

        # 새 토큰을 추가하지 않고 개행 문자 처리
        if "\n" not in self.inverse_vocab:
            # '\n'에 대한 플레이스홀더로 기존 토큰 ID 사용
            # 가능하다면 "<|endoftext|>"를 우선적으로 사용
            fallback_token = next((token for token in ["<|endoftext|>", "Ġ", ""] if token in self.inverse_vocab), None)
            if fallback_token is not None:
                newline_token_id = self.inverse_vocab[fallback_token]
            else:
                # 대체 토큰이 없으면 오류 발생
                raise KeyError("No suitable token found in vocabulary to map '\\n'.")

            self.inverse_vocab["\n"] = newline_token_id
            self.vocab[newline_token_id] = "\n"

        # GPT-2 병합을 로드하고 할당된 "순위"와 함께 저장
        self.bpe_ranks = {}  # 순위 재설정
        with open(bpe_merges_path, "r", encoding="utf-8") as file:
            lines = file.readlines()
            if lines and lines[0].startswith("#"):
                lines = lines[1:]

            rank = 0
            for line in lines:
                pair = tuple(line.strip().split())
                if len(pair) == 2:
                    token1, token2 = pair
                    # token1 또는 token2가 어휘에 없으면 건너뛰기
                    if token1 in self.inverse_vocab and token2 in self.inverse_vocab:
                        self.bpe_ranks[(token1, token2)] = rank
                        rank += 1
                    else:
                        print(f"Skipping pair {pair} as one token is not in the vocabulary.")

    def encode(self, text, allowed_special=None):
        """
        tiktoken 스타일의 특수 토큰 처리로 입력 텍스트를 토큰 ID 목록으로 인코딩합니다.
    
        Args:
            text (str): 인코딩할 입력 텍스트.
            allowed_special (set or None): 통과를 허용할 특수 토큰. None이면 특수 처리가 비활성화됩니다.
    
        Returns:
            토큰 ID 목록.
        """
        import re
    
        token_ids = []
    
        # 특수 토큰 처리가 활성화된 경우
        if allowed_special is not None and len(allowed_special) > 0:
            # 허용된 특수 토큰을 매치하는 정규식 구축
            special_pattern = (
                "(" + "|".join(re.escape(tok) for tok in sorted(allowed_special, key=len, reverse=True)) + ")"
            )
    
            last_index = 0
            for match in re.finditer(special_pattern, text):
                prefix = text[last_index:match.start()]
                token_ids.extend(self.encode(prefix, allowed_special=None))  # 특수 처리 없이 접두사 인코딩
    
                special_token = match.group(0)
                if special_token in self.inverse_vocab:
                    token_ids.append(self.inverse_vocab[special_token])
                else:
                    raise ValueError(f"Special token {special_token} not found in vocabulary.")
                last_index = match.end()
    
            text = text[last_index:]  # 정상적으로 처리할 나머지 부분
    
            # 나머지에서 허용되지 않은 특수 토큰이 있는지 확인
            disallowed = [
                tok for tok in self.inverse_vocab
                if tok.startswith("<|") and tok.endswith("|>") and tok in text and tok not in allowed_special
            ]
            if disallowed:
                raise ValueError(f"Disallowed special tokens encountered in text: {disallowed}")
    
        # 특수 토큰이 없거나, 특수 토큰 분할 후 남은 텍스트:
        tokens = []
        lines = text.split("\n")
        for i, line in enumerate(lines):
            if i > 0:
                tokens.append("\n")
            words = line.split()
            for j, word in enumerate(words):
                if j == 0 and i > 0:
                    tokens.append("Ġ" + word)
                elif j == 0:
                    tokens.append(word)
                else:
                    tokens.append("Ġ" + word)
    
        for token in tokens:
            if token in self.inverse_vocab:
                token_ids.append(self.inverse_vocab[token])
            else:
                token_ids.extend(self.tokenize_with_bpe(token))
    
        return token_ids

    def tokenize_with_bpe(self, token):
        """
        BPE 병합을 사용하여 단일 토큰을 토큰화합니다.

        Args:
            token (str): 토큰화할 토큰.

        Returns:
            List[int]: BPE 적용 후 토큰 ID 목록.
        """
        # 토큰을 개별 문자로 토큰화 (초기 토큰 ID로)
        token_ids = [self.inverse_vocab.get(char, None) for char in token]
        if None in token_ids:
            missing_chars = [char for char, tid in zip(token, token_ids) if tid is None]
            raise ValueError(f"Characters not found in vocab: {missing_chars}")

        # OpenAI의 GPT-2 병합을 로드하지 않은 경우, 제 접근법을 사용
        if not self.bpe_ranks:
            can_merge = True
            while can_merge and len(token_ids) > 1:
                can_merge = False
                new_tokens = []
                i = 0
                while i < len(token_ids) - 1:
                    pair = (token_ids[i], token_ids[i + 1])
                    if pair in self.bpe_merges:
                        merged_token_id = self.bpe_merges[pair]
                        new_tokens.append(merged_token_id)
                        # 교육 목적으로 주석 해제:
                        # print(f"Merged pair {pair} -> {merged_token_id} ('{self.vocab[merged_token_id]}')")
                        i += 2  # 병합되었으므로 다음 토큰 건너뛰기
                        can_merge = True
                    else:
                        new_tokens.append(token_ids[i])
                        i += 1
                if i < len(token_ids):
                    new_tokens.append(token_ids[i])
                token_ids = new_tokens
            return token_ids

        # 그렇지 않으면 순위를 사용한 GPT-2 스타일 병합 수행:
        # 1) token_ids를 각 ID에 대한 문자열 "기호"로 다시 변환
        symbols = [self.vocab[id_num] for id_num in token_ids]

        # 가장 낮은 순위 페어를 모든 발생에서 반복적으로 병합
        while True:
            # 모든 인접한 페어 수집
            pairs = set(zip(symbols, symbols[1:]))
            if not pairs:
                break

            # 가장 좋은(가장 낮은) 순위를 가진 페어 찾기
            min_rank = float("inf")
            bigram = None
            for p in pairs:
                r = self.bpe_ranks.get(p, float("inf"))
                if r < min_rank:
                    min_rank = r
                    bigram = p

            # 유효한 순위를 가진 페어가 없으면 완료
            if bigram is None or bigram not in self.bpe_ranks:
                break

            # 해당 페어의 모든 발생 병합
            first, second = bigram
            new_symbols = []
            i = 0
            while i < len(symbols):
                # 위치 i에서 (first, second)를 보면 병합
                if i < len(symbols) - 1 and symbols[i] == first and symbols[i+1] == second:
                    new_symbols.append(first + second)  # 병합된 기호
                    i += 2
                else:
                    new_symbols.append(symbols[i])
                    i += 1
            symbols = new_symbols

            if len(symbols) == 1:
                break

        # 마지막으로, 병합된 기호들을 ID로 다시 변환
        merged_ids = [self.inverse_vocab[sym] for sym in symbols]
        return merged_ids

    def decode(self, token_ids):
        """
        토큰 ID 목록을 다시 문자열로 디코딩합니다.

        Args:
            token_ids (List[int]): 디코딩할 토큰 ID 목록.

        Returns:
            str: 디코딩된 문자열.
        """
        decoded_string = ""
        for i, token_id in enumerate(token_ids):
            if token_id not in self.vocab:
                raise ValueError(f"Token ID {token_id} not found in vocab.")
            token = self.vocab[token_id]
            if token == "\n":
                if decoded_string and not decoded_string.endswith(" "):
                    decoded_string += " "  # 개행 앞에 공백이 없으면 공백 추가
                decoded_string += token
            elif token.startswith("Ġ"):
                decoded_string += " " + token[1:]
            else:
                decoded_string += token
        return decoded_string

    def save_vocab_and_merges(self, vocab_path, bpe_merges_path):
        """
        어휘와 BPE 병합을 JSON 파일로 저장합니다.

        Args:
            vocab_path (str): 어휘를 저장할 경로.
            bpe_merges_path (str): BPE 병합을 저장할 경로.
        """
        # 어휘 저장
        with open(vocab_path, "w", encoding="utf-8") as file:
            json.dump(self.vocab, file, ensure_ascii=False, indent=2)

        # BPE 병합을 딕셔너리 목록으로 저장
        with open(bpe_merges_path, "w", encoding="utf-8") as file:
            merges_list = [{"pair": list(pair), "new_id": new_id}
                           for pair, new_id in self.bpe_merges.items()]
            json.dump(merges_list, file, ensure_ascii=False, indent=2)

    def load_vocab_and_merges(self, vocab_path, bpe_merges_path):
        """
        JSON 파일에서 어휘와 BPE 병합을 로드합니다.

        Args:
            vocab_path (str): 어휘 파일 경로.
            bpe_merges_path (str): BPE 병합 파일 경로.
        """
        # 어휘 로드
        with open(vocab_path, "r", encoding="utf-8") as file:
            loaded_vocab = json.load(file)
            self.vocab = {int(k): v for k, v in loaded_vocab.items()}
            self.inverse_vocab = {v: int(k) for k, v in loaded_vocab.items()}

        # BPE 병합 로드
        with open(bpe_merges_path, "r", encoding="utf-8") as file:
            merges_list = json.load(file)
            for merge in merges_list:
                pair = tuple(merge["pair"])
                new_id = merge["new_id"]
                self.bpe_merges[pair] = new_id

    @lru_cache(maxsize=None)
    def get_special_token_id(self, token):
        return self.inverse_vocab.get(token, None)

    @staticmethod
    def find_freq_pair(token_ids, mode="most"):
        pairs = Counter(zip(token_ids, token_ids[1:]))

        if not pairs:
            return None

        if mode == "most":
            return max(pairs.items(), key=lambda x: x[1])[0]
        elif mode == "least":
            return min(pairs.items(), key=lambda x: x[1])[0]
        else:
            raise ValueError("Invalid mode. Choose 'most' or 'least'.")

    @staticmethod
    def replace_pair(token_ids, pair_id, new_id):
        dq = deque(token_ids)
        replaced = []

        while dq:
            current = dq.popleft()
            if dq and (current, dq[0]) == pair_id:
                replaced.append(new_id)
                # 페어의 두 번째 토큰 제거, 첫 번째는 이미 제거됨
                dq.popleft()
            else:
                replaced.append(current)

        return replaced

- 위의 `BPETokenizerSimple` 클래스에는 많은 코드가 있으며, 이를 자세히 논의하는 것은 이 노트북의 범위를 벗어나지만, 다음 섹션에서는 클래스 메소드를 좀 더 잘 이해하기 위한 사용법에 대한 간단한 개요를 제공합니다

## 3. BPE 구현 연습

- 실제로는 위의 제 구현이 성능이 아닌 가독성과 교육적 목적에 중점을 두고 있으므로 [tiktoken](https://github.com/openai/tiktoken)을 사용하는 것을 강력히 권장합니다
- 그러나 tiktoken에 훈련 메소드가 없다는 점을 제외하면 사용법은 tiktoken과 거의 유사합니다
- 위의 제 `BPETokenizerSimple` Python 코드가 어떻게 작동하는지 아래의 몇 가지 예제를 살펴보면서 알아봅시다(자세한 코드 논의는 이 노트북의 범위를 벗어남)

### 3.1 훈련, 인코딩, 디코딩

- 먼저, 훈련 데이터셋으로 샘플 텍스트를 고려해봅시다:

In [ ]:
import os
import urllib.request

def download_file_if_absent(url, filename, search_dirs):
    for directory in search_dirs:
        file_path = os.path.join(directory, filename)
        if os.path.exists(file_path):
            print(f"{filename} already exists in {file_path}")
            return file_path

    target_path = os.path.join(search_dirs[0], filename)
    try:
        with urllib.request.urlopen(url) as response, open(target_path, "wb") as out_file:
            out_file.write(response.read())
        print(f"Downloaded {filename} to {target_path}")
    except Exception as e:
        print(f"Failed to download {filename}. Error: {e}")
    return target_path

verdict_path = download_file_if_absent(
    url=(
         "https://raw.githubusercontent.com/rasbt/"
         "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
         "the-verdict.txt"
    ),
    filename="the-verdict.txt",
    search_dirs=["ch02/01_main-chapter-code/", "../01_main-chapter-code/", "."]
)

with open(verdict_path, "r", encoding="utf-8") as f: # ../01_main-chapter-code/ 추가됨
    text = f.read()

- 다음으로, 어휘 크기가 1,000인 BPE 토크나이저를 초기화하고 훈련해봅시다
- 앞에서 논의된 바이트 값으로 인해 어휘 크기가 기본적으로 이미 256이므로, 우리는 실제로 744개의 어휘 항목만을 "학습"합니다(`<|endoftext|>` 특수 토큰과 `Ġ` 공백 토큰을 고려하면 정확히는 742개입니다)
- 비교해보면, GPT-2 어휘는 50,257개 토큰, GPT-4 어휘는 100,256개 토큰(tiktoken에서 `cl100k_base`), GPT-4o는 199,997개 토큰(tiktoken에서 `o200k_base`)을 사용합니다. 이들은 모두 위의 간단한 예제 텍스트와 비교하여 훨씬 더 큰 훈련 세트를 가지고 있습니다

In [ ]:
tokenizer = BPETokenizerSimple()
tokenizer.train(text, vocab_size=1000, allowed_special={"<|endoftext|>"})

- 어휘 내용을 검사하고 싶을 수도 있습니다(하지만 긴 목록이 생성될 것입니다)

In [ ]:
# print(tokenizer.vocab)
print(len(tokenizer.vocab))

- 이 어휘는 742번의 병합으로 생성됩니다(`= 1000 - len(range(0, 256)) - len(special_tokens) - "Ġ" = 1000 - 256 - 1 - 1 = 742`)

In [ ]:
print(len(tokenizer.bpe_merges))

- 이는 첫 256개 항목이 단일 문자 토큰이라는 것을 의미합니다

- 다음으로, `encode` 메소드를 통해 생성된 병합을 사용하여 일부 텍스트를 인코딩해봅시다:

In [ ]:
input_text = "Jack embraced beauty through art and life."
token_ids = tokenizer.encode(input_text)
print(token_ids)

In [ ]:
input_text = "Jack embraced beauty through art and life.<|endoftext|> "
token_ids = tokenizer.encode(input_text)
print(token_ids)

In [ ]:
input_text = "Jack embraced beauty through art and life.<|endoftext|> "
token_ids = tokenizer.encode(input_text, allowed_special={"<|endoftext|>"})
print(token_ids)

In [ ]:
print("문자 수(Number of characters):", len(input_text))
print("토큰 ID 수(Number of token IDs):", len(token_ids))

- 위의 길이에서 볼 수 있듯이, 42자 문장이 20개의 토큰 ID로 인코딩되어, 문자-바이트 기반 인코딩과 비교하여 입력 길이를 대략 절반으로 줄였습니다

- 어휘 자체는 토큰 ID를 다시 텍스트로 매핑할 수 있게 해주는 `decode()` 메소드에서 사용됩니다:

In [ ]:
print(token_ids)

In [ ]:
print(tokenizer.decode(token_ids))

- 각 토큰 ID를 반복하면 토큰 ID가 어휘를 통해 어떻게 디코딩되는지 더 잘 이해할 수 있습니다:

In [ ]:
for token_id in token_ids:
    print(f"{token_id} -> {tokenizer.decode([token_id])}")

- 보시다시피, 대부분의 토큰 ID는 2문자 부분 단어를 나타냅니다. 이는 훈련 데이터 텍스트가 매우 짧고 반복적인 단어가 그리 많지 않으며, 상대적으로 작은 어휘 크기를 사용했기 때문입니다

- 요약하면, `decode(encode())`를 호출하면 임의의 입력 텍스트를 재현할 수 있어야 합니다:

In [ ]:
tokenizer.decode(
    tokenizer.encode("This is some text.")
)

In [ ]:
tokenizer.decode(
    tokenizer.encode("This is some text with \n newline characters.")
)

### 3.2 토크나이저 저장 및 로딩

- 다음으로, 훈련된 토크나이저를 나중에 재사용하기 위해 저장하는 방법을 살펴봅시다:

In [ ]:
# 훈련된 토크나이저 저장
tokenizer.save_vocab_and_merges(vocab_path="vocab.json", bpe_merges_path="bpe_merges.txt")

In [ ]:
# 토크나이저 로드
tokenizer2 = BPETokenizerSimple()
tokenizer2.load_vocab_and_merges(vocab_path="vocab.json", bpe_merges_path="bpe_merges.txt")

- 로드된 토크나이저는 이전과 동일한 결과를 생성할 수 있어야 합니다:

In [ ]:
print(tokenizer2.decode(token_ids))

In [ ]:
tokenizer2.decode(
    tokenizer2.encode("This is some text with \n newline characters.")
)

&nbsp;
### 3.3 OpenAI의 원본 GPT-2 BPE 토크나이저 로딩

- 마지막으로, OpenAI의 GPT-2 토크나이저 파일을 로드해봅시다

In [ ]:
# 이 디렉토리에 아직 없으면 파일 다운로드

# 검색할 디렉토리와 다운로드할 파일 정의
search_directories = ["ch02/02_bonus_bytepair-encoder/gpt2_model/", "../02_bonus_bytepair-encoder/gpt2_model/", "."]

files_to_download = {
    "https://openaipublic.blob.core.windows.net/gpt-2/models/124M/vocab.bpe": "vocab.bpe",
    "https://openaipublic.blob.core.windows.net/gpt-2/models/124M/encoder.json": "encoder.json"
}

# 디렉토리가 존재하는지 확인하고 필요시 파일 다운로드
paths = {}
for url, filename in files_to_download.items():
    paths[filename] = download_file_if_absent(url, filename, search_directories)

- 다음으로, `load_vocab_and_merges_from_openai` 메소드를 통해 파일을 로드합니다:

In [ ]:
tokenizer_gpt2 = BPETokenizerSimple()
tokenizer_gpt2.load_vocab_and_merges_from_openai(
    vocab_path=paths["encoder.json"], bpe_merges_path=paths["vocab.bpe"]
)

- 어휘 크기는 아래 코드로 확인할 수 있듯이 `50257`이어야 합니다:

In [ ]:
len(tokenizer_gpt2.vocab)

- 이제 `BPETokenizerSimple` 객체를 통해 GPT-2 토크나이저를 사용할 수 있습니다:

In [ ]:
input_text = "This is some text"
token_ids = tokenizer_gpt2.encode(input_text)
print(token_ids)

In [ ]:
print(tokenizer_gpt2.decode(token_ids))

- 상호작용형 [tiktoken 앱](https://tiktokenizer.vercel.app/?model=gpt2)이나 [tiktoken 라이브러리](https://github.com/openai/tiktoken)를 사용하여 올바른 토큰을 생성하는지 다시 확인할 수 있습니다:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/bpe-from-scratch/tiktokenizer.webp" width="600px">

```python
import tiktoken

gpt2_tokenizer = tiktoken.get_encoding("gpt2")
gpt2_tokenizer.encode("This is some text")
# prints [1212, 318, 617, 2420]
```

&nbsp;
# 4. 결론

- 바로 이것입니다! 새로운 토크나이저를 만들기 위한 훈련 방법이나 원본 OpenAI GPT-2 모델에서 GPT-2 토크나이저 어휘와 병합을 로드하는 방법과 함께 BPE가 간단히 작동하는 방식입니다
- 이 간단한 튜토리얼이 교육 목적으로 유용하기를 바랍니다. 질문이 있으시면 [여기](https://github.com/rasbt/LLMs-from-scratch/discussions/categories/q-a)에서 새로운 토론을 자유롭게 열어주세요
- 다른 토크나이저 구현과의 성능 비교는 [이 노트북](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch02/02_bonus_bytepair-encoder/compare-bpe-tiktoken.ipynb)을 참조하세요